# BART News Summarization (ILSUM English)
Colab-compatible notebook.
Run install cell, then restart runtime.

In [2]:
!pip install -q transformers==4.46.0 datasets==2.19.1 accelerate==0.34.0 evaluate rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 114.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==20

Restart runtime before continuing.

In [1]:
from datasets import load_dataset
from transformers import BartTokenizer, BartForConditionalGeneration
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import evaluate

In [21]:
models_info = [
    ("BERT", "512 tokens", "110M", "Classification, QA"),
    ("GPT-2", "1024 tokens", "117M", "Text generation"),
    ("T5", "512-1024 tokens", "220M+", "Text-to-text tasks"),
    ("BART-base", "1024 tokens", "140M", "Summarization"),
    ("BART-large", "1024 tokens", "406M", "High-quality summarization")
]

for m in models_info:
    print(m)

('BERT', '512 tokens', '110M', 'Classification, QA')
('GPT-2', '1024 tokens', '117M', 'Text generation')
('T5', '512-1024 tokens', '220M+', 'Text-to-text tasks')
('BART-base', '1024 tokens', '140M', 'Summarization')
('BART-large', '1024 tokens', '406M', 'High-quality summarization')


In [2]:
dataset = load_dataset('ILSUM/ILSUM-1.0', 'English')
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/12565 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4487 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/898 [00:00<?, ? examples/s]

{'id': '3938f547c863630032649c54e611e6b0', 'Article': 'Logos for MasterCard and Visa credit cards at the entrance of a New York coffee shopIn the latest blow to Russia’s financial system after its invasion of Ukraine, Mastercard and Visa said they are suspending their operations in the country. Mastercard said cards issued by Russian banks will no longer be supported by its network and any Mastercard issued outside the country will not work at Russian stores or ATMs.“We don’t take this decision lightly,” Mastercard said in a statement, adding that it made the move after discussions with customers, partners and governments.Visa said it’s working with clients and partners in Russia to cease all Visa transactions over the coming days.“We are compelled to act following Russia’s unprovoked invasion of Ukraine, and the unacceptable events that we have witnessed,” Visa Chairman and Chief Executive Officer Al Kelly said in a statement.The twin suspensions were announced within 16 minutes of ea

In [3]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [4]:
def filter_data(example):
    return len(example['Article'].split()) <= 400 and len(example['Summary'].split()) <= 100

dataset = dataset.filter(filter_data)

Filter:   0%|          | 0/12565 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4487 [00:00<?, ? examples/s]

Filter:   0%|          | 0/898 [00:00<?, ? examples/s]

In [22]:
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 6130
    })
    test: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 2136
    })
    validation: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 442
    })
})
{'id': '1b336d62e9502c5b91b7afd3c7bcff46', 'Article': 'Author-Diplomat Vikas Swarup moved to Delhi as Indo-Canadian ties remain coldAuthor-diplomat Vikas Swarup, India\'s current High Commissioner to Canada, has been appointed as Secretary in the Consular, Passport, Visa and Overseas Indian Affairs division, with effect from August 1.According to a circular issued by the Department of Personnel and Training, "The Appointments Committee of the Cabinet has approved the appointment of Vikas Swarup, (Indian Foreign Service officer of 1986 batch) High Commissioner in Ottawa, as Secretary (CPV &OIA) in the Ministry of External Affairs with effect f

In [5]:
def preprocess(example):
    inputs = tokenizer(example['Article'], max_length=512, truncation=True, padding='max_length')
    targets = tokenizer(example['Summary'], max_length=128, truncation=True, padding='max_length')
    inputs['labels'] = targets['input_ids']
    return inputs

tokenized_dataset = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/6130 [00:00<?, ? examples/s]

Map:   0%|          | 0/2136 [00:00<?, ? examples/s]

Map:   0%|          | 0/442 [00:00<?, ? examples/s]

In [6]:
model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

In [7]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [8]:
rouge = evaluate.load('rouge')
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return rouge.compute(predictions=decoded_preds, references=decoded_labels)

In [15]:
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=2,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=100,
    fp16=True,
    eval_accumulation_steps=1,
    prediction_loss_only=True)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_5746/3948393196.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.066500,0.111455
2,0.052200,0.115831


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:2816: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=1534, training_loss=0.06456005114775437, metrics={'train_runtime': 419.329, 'train_samples_per_second': 29.237, 'train_steps_per_second': 3.658, 'total_flos': 3737684489011200.0, 'train_loss': 0.06456005114775437, 'epoch': 2.0})

In [18]:
results = trainer.evaluate(tokenized_dataset['test'])
print(results)

{'eval_loss': 0.10747385025024414, 'eval_runtime': 18.7223, 'eval_samples_per_second': 114.089, 'eval_steps_per_second': 14.261, 'epoch': 2.0}


In [19]:
def generate_summary(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=30,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True,
        no_repeat_ngram_size=3
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [20]:
for i in range(5):
    sample = dataset["test"][i]

    article = sample["Article"]
    actual  = sample["Summary"]

    pred = generate_summary(article)

    print("\n==============================")
    print("Article:\n", article[:300])
    print("\nGenerated Summary:\n", pred)
    print("\nActual Summary:\n", actual)


Article:
 Indian-origin boy finds millions of years old fossil in UK gardenA six-year-old Indian-origin boy says he is “really excited” after he found a fossil from millions of years ago while digging in his garden in the West Midlands region of England. Siddak Singh Jhamat, known as Sid, was using a fossil-h

Generated Summary:
 A six-year-old Indian-origin boy says he is “really excited” after he found a fossil from millions of years ago while digging in his garden in the West Midlands region of England.

Actual Summary:
 Siddak Singh Jhamat, known as Sid, was using a fossil-hunting set he received as a Christmas present when he came across a rock that looked like a horn.

Article:
 Representative ImageA 38-year-old Indian man has been charged with conspiring to smuggle six of his countrymen into the US on commercial airline flights, authorities said.Bhavin Patel was arrested at the Newark Liberty International Airport last week and was charged on Monday with one count of consp

Gen